# SVTR Model

## DEPENDENCIES & LIBRARIES

In [2]:
# 1. Mount Google Drive to access your files
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone the PaddleOCR repository
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR

# 3. Install dependencies from the requirements file
!pip install -r requirements.txt

# 4. Install paddlepaddle-gpu if not already installed
# Google Colab GPU runtimes have CUDA pre-installed.
!pip install paddlepaddle

# 5. Run the conversion script
# Replace the paths with your own file locations.
# The paths are relative to your Google Drive mounted folder.


Mounted at /content/drive
Cloning into 'PaddleOCR'...
remote: Enumerating objects: 282060, done.
remote: Counting objects: 100% (2520/2520), done.
remote: Compressing objects: 100% (526/526), done.
remote: Total 282060 (delta 2200), reused 2079 (delta 1988), pack-reused 279540 (from 3)
Receiving objects: 100% (282060/282060), 1.49 GiB | 24.70 MiB/s, done.
Resolving deltas: 100% (222752/222752), done.
Updating files: 100% (1963/1963), done.
/content/PaddleOCR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.8/963.8 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.0/303.0 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: opt_einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully u

In [19]:
!python3 /content/PaddleOCR/tools/export_model.py \
    -c /content/drive/MyDrive/config.yml \
    -o Global.pretrained_model=/content/drive/MyDrive/best_accuracy.pdparams \
    Global.save_inference_dir=/content/drive/MyDrive/inference_model/

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:717: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
[2025/09/08 09:54:05] ppocr INFO: load pretrain successful from /content/drive/MyDrive/best_accuracy
[2025/09/08 09:54:05] ppocr INFO: Export inference config file to /content/drive/MyDrive/inference_model/inference.yml
Skipping import of the encryption module
Traceback (most recent call last):
  File "/content/PaddleOCR/tools/export_model.py", line 37, in <module>
    main()
  File "/content/PaddleOCR/tools/export_model.py", line 33, in main
    export(config)
  File "/content/PaddleOCR/ppocr/utils/export_model.py", line 543, in export
    export_single_model(
  File "/content/PaddleOCR/ppocr/utils/export_model.py", line 394, 

In [8]:
import os
import subprocess
import urllib.request
import tarfile
from glob import glob
from typing import List, Tuple, Dict, Optional
import cv2
import numpy as np
from tqdm import tqdm
from collections import Counter
from pathlib import Path
import yaml
import re
import shutil
import random
import threading
import albumentations as A
from albumentations import Compose


# --- Step 2: Download the Pretrained SVTR Model and Dictionary ---
def download_pretrained_model():
    """
    Download PP-OCRv3 English recognition model (SVTR-based)
    """
    model_config = {
        'name': 'PP-OCRv3 English Recognition',
        'url': 'https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_rec_train.tar',
        'extract_dir': '/content/PaddleOCR/pretrain_models/en_ppocr_v3_rec/',
    }

    try:
        # Create directory
        os.makedirs(model_config['extract_dir'], exist_ok=True)

        # Check if model already exists
        model_files = list(Path(model_config['extract_dir']).rglob('*.pdparams'))
        if model_files:
            print(f"Pretrained model already exists at: {model_files[0]}")
            return str(model_files[0])

        # Download
        temp_file = "/tmp/pretrained_model.tar"
        print(f"Downloading {model_config['name']} from {model_config['url']}...")
        urllib.request.urlretrieve(model_config['url'], temp_file)

        # Extract
        print(f"Extracting to {model_config['extract_dir']}...")
        with tarfile.open(temp_file, 'r') as tar:
            tar.extractall(model_config['extract_dir'])

        # Clean up temp file
        os.remove(temp_file)

        # Find the model file
        model_files = list(Path(model_config['extract_dir']).rglob('*.pdparams'))

        if model_files:
            model_path = str(model_files[0])
            print(f"✅ Successfully downloaded model to {model_path}")
            return model_path
        else:
            print(f"❌ No model files found after extraction")
            return None

    except Exception as e:
        print(f"❌ Failed to download pretrained model: {str(e)}")
        return None

## DATA PREPROCESSING & LOADING

In [16]:
def get_character_dict_path():
    """
    Get the path to the English character dictionary
    """
    # Try to find the dictionary that comes with the pretrained model
    pretrain_dir = Path('/content/PaddleOCR/pretrain_models/')
    dict_files = list(pretrain_dir.rglob('en_dict.txt'))

    if dict_files:
        return str(dict_files[0])

    # Use PaddleOCR's built-in English dictionary
    paddleocr_dict = '/content/PaddleOCR/ppocr/utils/en_dict.txt'
    if os.path.exists(paddleocr_dict):
        return paddleocr_dict

    # Create a comprehensive English dictionary
    dict_path = '/content/PaddleOCR/pretrain_models/en_dict_comprehensive.txt'
    chars = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~ "

    os.makedirs(os.path.dirname(dict_path), exist_ok=True)
    with open(dict_path, 'w', encoding='utf-8') as f:
        for char in chars:
            f.write(char + '\n')

    print(f"Created comprehensive English dictionary at: {dict_path}")
    return dict_path

# --- Step 3: Data Processing Class and Function ---
class DataProcessor:
    """
    Enhanced data processor for SROIE receipt text recognition (Warning-free version)
    """
    def __init__(self, target_height=32, min_width=8, max_width=320, augment_multiplier=3):
        self.target_height = target_height
        self.min_width = min_width
        self.max_width = max_width
        self.augment_multiplier = augment_multiplier
        self.augmentations = self._create_augmentation_pipeline()

    def _create_augmentation_pipeline(self):
        """Create warning-free augmentation pipeline"""
        return Compose([
            # Geometric augmentations
            A.OneOf([
                A.Rotate(limit=2, p=0.4),
                A.Perspective(scale=(0.02, 0.05), p=0.3),
                A.Affine(
                    translate_percent=0.02,
                    scale=(0.95, 1.05),
                    rotate=2,
                    p=0.3
                ),
            ], p=0.6),

            # Blur effects
            A.OneOf([
                A.GaussianBlur(blur_limit=3, p=0.4),
                A.MotionBlur(blur_limit=5, p=0.3),
            ], p=0.3),

            # Noise
            A.OneOf([
                A.GaussNoise(var_limit=10, p=0.4),
                A.ISONoise(p=0.3),
            ], p=0.4),

            # Brightness and contrast
            A.OneOf([
                A.RandomBrightnessContrast(
                    brightness_limit=0.1,
                    contrast_limit=0.1,
                    p=0.5
                ),
                A.CLAHE(clip_limit=2.0, p=0.3),
                A.RandomGamma(gamma_limit=(90, 110), p=0.3),
            ], p=0.5),

            # Color adjustments
            A.OneOf([
                A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=10, val_shift_limit=10, p=0.3),
                A.RGBShift(r_shift_limit=10, g_shift_limit=10, b_shift_limit=10, p=0.3),
            ], p=0.3),

        ], p=0.8)

    def preprocess_image(self, image: np.ndarray) -> np.ndarray:
        """
        Preprocess image for better OCR recognition
        """
        # Convert to grayscale if needed
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image.copy()

        # Apply adaptive thresholding for better text contrast
        binary1 = cv2.adaptiveThreshold(
            gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 23, 5
        )

        _, binary2 = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(gray)
        _, binary3 = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        binary = binary1

        # Noise reduction with different kernel sizes
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 1))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

        # Convert back to 3-channel for compatibility
        return cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)

    def resize_with_aspect_ratio(self, image: np.ndarray) -> np.ndarray:
        """
        Resize image while maintaining aspect ratio
        """
        h, w = image.shape[:2]

        # Calculate new width based on target height
        aspect_ratio = w / h
        new_width = int(self.target_height * aspect_ratio)

        # Ensure width is within bounds
        new_width = max(self.min_width, min(new_width, self.max_width))

        # Resize image
        interpolations = [cv2.INTER_CUBIC, cv2.INTER_LINEAR, cv2.INTER_AREA]
        resized = cv2.resize(image, (new_width, self.target_height),
                           interpolation=random.choice(interpolations))

        return resized

    def apply_augmentation(self, image: np.ndarray) -> List[np.ndarray]:
        """
        Apply offline augmentation to create multiple versions
        """
        augmented_images = [image.copy()]  # Original image

        # Generate multiple augmented versions
        for i in range(self.augment_multiplier):
            try:
                # Apply augmentation pipeline
                augmented = self.augmentations(image=image)['image']
                augmented_images.append(augmented)
            except Exception as e:
                print(f"Augmentation failed: {e}, using manual augmentation")
                # If augmentation fails, create slight variations manually
                augmented = self._manual_augmentation(image)
                augmented_images.append(augmented)

        return augmented_images

    def _manual_augmentation(self, image: np.ndarray) -> np.ndarray:
        """
        Fallback manual augmentation methods
        """
        h, w = image.shape[:2]
        augmented = image.copy()

        # Random brightness adjustment
        brightness_factor = random.uniform(0.8, 1.2)
        augmented = cv2.convertScaleAbs(augmented, alpha=brightness_factor, beta=0)

        # Random slight rotation
        angle = random.uniform(-1, 1)
        center = (w // 2, h // 2)
        rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
        augmented = cv2.warpAffine(augmented, rotation_matrix, (w, h),
                                 borderMode=cv2.BORDER_REPLICATE)

        # Random noise
        noise = np.random.normal(0, 5, augmented.shape).astype(np.uint8)
        augmented = cv2.add(augmented, noise)

        return augmented

    def validate_and_clean_crop(self, image: np.ndarray, coords: List[int],
                               text_label: str, enable_augmentation: bool = True,
                               debug: bool = False) -> Tuple[bool, List[np.ndarray], str]:
        """
        Enhanced validation and cleaning for crops with debugging
        """
        # Clean text label
        original_label = text_label
        text_label = text_label.strip()

        if debug:
            print(f"Debug - Original label: '{original_label}' -> Cleaned: '{text_label}'")

        if not text_label or len(text_label) < 1:
            if debug:
                print("Debug - Rejected: Empty text label")
            return False, None, ""

        # Remove non-printable characters but keep spaces
        text_label = ''.join(char for char in text_label if char.isprintable())
        text_label = text_label.strip()

        if not text_label:
            if debug:
                print("Debug - Rejected: Text label empty after cleaning")
            return False, None, ""

        # Get bounding box
        try:
            x_coords = coords[0::2]
            y_coords = coords[1::2]
            x_min, y_min = max(0, min(x_coords)), max(0, min(y_coords))
            x_max, y_max = min(image.shape[1], max(x_coords)), min(image.shape[0], max(y_coords))
        except Exception as e:
            if debug:
                print(f"Debug - Rejected: Error parsing coordinates: {e}")
            return False, None, text_label

        width, height = x_max - x_min, y_max - y_min

        if debug:
            print(f"Debug - Dimensions: {width}x{height}, Coords: ({x_min},{y_min}) to ({x_max},{y_max})")

        # More lenient validation - relaxed constraints
        if width < 4 or height < 4:  # Reduced minimum size
            if debug:
                print(f"Debug - Rejected: Too small ({width}x{height})")
            return False, None, text_label

        if width > 2000 or height > 500:  # Increased maximum size
            if debug:
                print(f"Debug - Rejected: Too large ({width}x{height})")
            return False, None, text_label

        # Add padding
        padding = 2  # Reduced padding
        y_min = max(0, y_min - padding)
        y_max = min(image.shape[0], y_max + padding)
        x_min = max(0, x_min - padding)
        x_max = min(image.shape[1], x_max + padding)

        # Crop image
        cropped_img = image[y_min:y_max, x_min:x_max]

        if cropped_img.size == 0:
            if debug:
                print("Debug - Rejected: Empty crop")
            return False, None, text_label

        if enable_augmentation:
            # Get multiple augmented versions
            try:
                augmented_crops = self.apply_augmentation(cropped_img)
            except Exception as e:
                if debug:
                    print(f"Debug - Augmentation failed: {e}, using original only")
                augmented_crops = [cropped_img]
        else:
            # Just use original crop
            augmented_crops = [cropped_img]

        # Process each augmented crop
        processed_crops = []
        for crop in augmented_crops:
            try:
                # Preprocess the cropped image
                processed_img = self.preprocess_image(crop)
                # Resize with aspect ratio
                final_img = self.resize_with_aspect_ratio(processed_img)
                processed_crops.append(final_img)
            except Exception as e:
                if debug:
                    print(f"Debug - Processing failed for one crop: {e}")
                continue

        if not processed_crops:
            if debug:
                print("Debug - Rejected: No processed crops")
            return False, None, text_label

        if debug:
            print(f"Debug - SUCCESS: Created {len(processed_crops)} processed crops")

        return True, processed_crops, text_label

def data_loader(image_list: List[str], source_box_paths: List[str],
                dest_img_path: str, output_label_file: str,
                enable_augmentation: bool = True, augment_multiplier: int = 3,
                debug_mode: bool = False):
    """
    Enhanced data loader for SROIE dataset with debugging
    """
    processor = DataProcessor(augment_multiplier=augment_multiplier)

    # Create mapping from all possible box directories
    box_files = {}
    for box_path in source_box_paths:
        if os.path.exists(box_path):
            files = {os.path.splitext(os.path.basename(f))[0]: f
                    for f in glob(os.path.join(box_path, '*.txt'))}
            box_files.update(files)

    if debug_mode:
        print(f"Found {len(box_files)} box files")

    rec_data = []
    stats = {'processed': 0, 'valid': 0, 'skipped_no_box': 0, 'skipped_invalid': 0, 'augmented_crops': 0}

    # Create destination directory
    os.makedirs(dest_img_path, exist_ok=True)

    for img_idx, img_path in enumerate(tqdm(image_list, desc=f"Processing {os.path.basename(dest_img_path)} data")):
        base_name = os.path.splitext(os.path.basename(img_path))[0]

        if base_name not in box_files:
            if debug_mode:
                print(f"Debug - No box file for {base_name}")
            stats['skipped_no_box'] += 1
            continue

        try:
            image = cv2.imread(img_path)
            if image is None:
                if debug_mode:
                    print(f"Debug - Could not load image {img_path}")
                stats['skipped_invalid'] += 1
                continue

            box_path = box_files[base_name]

            with open(box_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()

            if debug_mode and img_idx == 0:  # Debug first image only
                print(f"Debug - Processing first image: {img_path}")
                print(f"Debug - Box file: {box_path}")
                print(f"Debug - Number of lines: {len(lines)}")

            for line_idx, line in enumerate(lines):
                stats['processed'] += 1
                parts = line.strip().split(',', 8)

                if len(parts) < 9:
                    if debug_mode and img_idx == 0 and line_idx < 5:
                        print(f"Debug - Line {line_idx}: insufficient parts ({len(parts)}): '{line.strip()}'")
                    stats['skipped_invalid'] += 1
                    continue

                try:
                    coords = [int(p) for p in parts[:8]]
                    text_label = parts[8].strip()

                    # Only debug first few crops of first image
                    debug_this_crop = debug_mode and img_idx == 0 and line_idx < 5

                    is_valid, cropped_imgs, clean_label = processor.validate_and_clean_crop(
                        image, coords, text_label, enable_augmentation, debug=debug_this_crop)

                    if not is_valid or not cropped_imgs:
                        stats['skipped_invalid'] += 1
                        continue

                    # Save all augmented versions
                    for aug_idx, cropped_img in enumerate(cropped_imgs):
                        cropped_img_name = f"{base_name}_{line_idx:03d}_aug{aug_idx:02d}.jpg"
                        cropped_img_path = os.path.join(dest_img_path, cropped_img_name)

                        # Save with high quality
                        cv2.imwrite(cropped_img_path, cropped_img, [cv2.IMWRITE_JPEG_QUALITY, 95])

                        relative_path = os.path.join(os.path.basename(dest_img_path), cropped_img_name)
                        rec_data.append(f"{relative_path}\t{clean_label}\n")
                        stats['augmented_crops'] += 1

                    stats['valid'] += 1

                except Exception as e:
                    if debug_mode and img_idx == 0 and line_idx < 5:
                        print(f"Debug - Exception processing line {line_idx}: {e}")
                    stats['skipped_invalid'] += 1
                    continue

        except Exception as e:
            if debug_mode:
                print(f"Debug - Exception processing image {img_path}: {e}")
            stats['skipped_invalid'] += 1
            continue

    # Write annotation file
    with open(output_label_file, 'w', encoding='utf-8') as f:
        f.writelines(rec_data)

    print(f"Processing stats for {os.path.basename(dest_img_path)}:")
    print(f"  Total text regions processed: {stats['processed']}")
    print(f"  Valid crops created: {stats['valid']}")
    print(f"  Total augmented crops: {stats['augmented_crops']}")
    print(f"  Images without box files: {stats['skipped_no_box']}")
    print(f"  Invalid/skipped regions: {stats['skipped_invalid']}")
    print(f"  Success rate: {stats['valid']/max(stats['processed'], 1)*100:.1f}%")

    return stats['valid']


# --- Step 4: Data Quality and Training Utility Functions ---

def check_data_quality(data_dir):
    """
    Check for potential data issues that cause overfitting
    """
    # Check training data
    train_gt_path = os.path.join(data_dir, 'train', 'train_gt.txt')
    val_gt_path = os.path.join(data_dir, 'val', 'val_gt.txt')

    # Check for both old and new names
    train_label_file = os.path.join(data_dir, 'train', 'train_rec.txt')
    val_label_file = os.path.join(data_dir, 'val', 'val_rec.txt')

    if not os.path.exists(train_label_file):
        train_label_file = train_gt_path
    if not os.path.exists(val_label_file):
        val_label_file = val_gt_path

    train_labels = []
    val_labels = []

    # Read training labels
    if os.path.exists(train_label_file):
        with open(train_label_file, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    train_labels.append(parts[1])

    # Read validation labels
    if os.path.exists(val_label_file):
        with open(val_label_file, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    val_labels.append(parts[1])

    # Check for data leakage
    train_set = set(train_labels)
    val_set = set(val_labels)
    overlap = train_set.intersection(val_set)

    print(f"Unique training labels: {len(train_set)}")
    print(f"Unique validation labels: {len(val_set)}")
    print(f"Overlapping labels: {len(overlap)} ({len(overlap)/len(val_set)*100:.1f}% of val)")

    # Check label distribution
    train_counter = Counter(train_labels)
    most_common_train = train_counter.most_common(10)

    print(f"\nMost common training labels:")
    for label, count in most_common_train:
        print(f"  '{label}': {count} times")

    # Check if labels are too simple
    avg_length = sum(len(label) for label in train_labels) / max(len(train_labels), 1)
    print(f"\nAverage label length: {avg_length:.1f}")

    # Check character diversity
    all_chars = set(''.join(train_labels))
    print(f"Unique characters in dataset: {len(all_chars)}")
    print(f"Characters: {''.join(sorted(all_chars))}")

    return {
        'train_count': len(train_labels),
        'val_count': len(val_labels),
        'overlap_count': len(overlap),
        'avg_length': avg_length,
        'unique_chars': len(all_chars)
    }

## EARLY STOPPER

In [15]:
class EarlyStoppingChecker:
    """
    Check if model is overfitting and should stop early
    """
    def __init__(self, patience=5, min_delta=0.01):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.counter = 0

    def check(self, val_score, train_score=None):
        """
        Check if training should stop
        """
        # Check for improvement in validation accuracy
        if self.best_score is None:
            self.best_score = val_score
            return False

        if val_score > self.best_score + self.min_delta:
            self.best_score = val_score
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
            return False

## MODEL

### Deeper architect

change Adam -> AdamW
Cosine LR to Piecewise

In [14]:
def create_svtr_config(work_dir: str, data_dir: str, pretrained_model_path: str, char_dict_path: str, checkpoint_path: str):
    """Create SVTR configuration with overfitting prevention measures"""
    config = {
        "Global": {
            "debug": False,
            "use_gpu": True,
            "epoch_num": 100,
            "log_smooth_window": 20,
            "print_batch_step": 50,
            "save_model_dir": f"{work_dir}/output/rec_svtr",
            "save_epoch_step": 3,
            "eval_batch_step": [0, 500],
            "cal_metric_during_train": True,
            "pretrained_model": pretrained_model_path,
            "checkpoints": checkpoint_path, #CONTINUE TRAINING
            "save_inference_dir": f"{work_dir}/output/rec_svtr_infer",
            "use_visualdl": True,
            "infer_img": "./doc/imgs_words/en/word_1.jpg",
            "character_dict_path": char_dict_path,
            "character_type": "en",
            "max_text_length": 25,
            "infer_mode": False,
            "use_space_char": True,
            "distributed": False,
            "save_res_path": f"{work_dir}/output/rec_svtr/predicts.txt"
        },
        "Optimizer": {
            "name": "AdamW",
            "beta1": 0.9,
            "beta2": 0.999,
            "weight_decay": 0.07,
            "lr": {
                "name": "Piecewise",
                "learning_rate": 0.0001,
                "decay_epochs": [45, 60, 80],
                "values": [0.0001, 0.00005, 0.00002, 0.00001],
                "warmup_epoch": 0
            }
        },
        "Architecture": {
            "model_type": "rec",
            "algorithm": "SVTR_LCNet",
            "Transform": None,
            "Backbone": {
                "name": "SVTRNet",
                "img_size": [32, 320],
                "out_channels": 128,
                "patch_merging": "Conv",
                "embed_dim": [64,128,256],
                "depth":  [3,6,3],
                "num_heads": [4,8,16],
                "mixer": ["Local"] * 6 + ["Global"] * 6,
                "local_mixer": [[7, 11], [7, 11], [7, 11]],
                "last_stage": True,
                "prenorm": False,
                "drop_rate": 0.02,
                "drop_path": 0.02
            },
            "Neck": {
                "name": "SequenceEncoder",
                "encoder_type": "reshape"
            },
            "Head": {
                "name": "CTCHead",
                "fc_decay": 1e-03
            }
        },
        "Loss": {
            "name": "CTCLoss"
        },
        "PostProcess": {
            "name": "CTCLabelDecode"
        },
        "Metric": {
            "name": "RecMetric",
            "main_indicator": "acc"
        },
        "Train": {
            "dataset": {
                "name": "SimpleDataSet",
                "data_dir": f"{data_dir}/train",
                "label_file_list": [f"{data_dir}/train/train_gt.txt"],
                "transforms": [
                    {"DecodeImage": {"img_mode": "BGR", "channel_first": False}},
                    {"CTCLabelEncode": {}},
                    {
                        "RecAug": {
                            "use_tia": False,
                            "aug_prob": 0.4
                        }
                    },
                    {"SVTRRecResizeImg": {"image_shape": [3, 48, 320], "padding": False} },
                    {"KeepKeys": {"keep_keys": ["image", "label", "length"]}}
                ]
            },
            "loader": {
                "shuffle": True,
                "batch_size_per_card": 80,
                "drop_last": True,
                "num_workers": 48
            }
        },
        "Eval": {
            "dataset": {
                "name": "SimpleDataSet",
                "data_dir": f"{data_dir}/val",
                "label_file_list": [f"{data_dir}/val/val_gt.txt"],
                "transforms": [
                    {"DecodeImage": {"img_mode": "BGR", "channel_first": False}},
                    {"CTCLabelEncode": {}},
                    {"SVTRRecResizeImg": {"image_shape": [3, 48, 320], "padding": False}},
                    {"KeepKeys": {"keep_keys": ["image", "label", "length"]}}
                ]
            },
            "loader": {
                "shuffle": False,
                "drop_last": False,
                "batch_size_per_card": 80,
                "num_workers": 48
            }
        }
    }

    config_path = f"{work_dir}/config.yml"
    os.makedirs(work_dir, exist_ok=True)

    with open(config_path, 'w', encoding='utf-8') as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

    return config_path

## Logger

In [13]:
class TrainingLogger:
    """Class to log and plot training metrics"""

    def __init__(self):
        self.epochs = []
        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []
        self.current_epoch = 0

    def update_epoch(self, epoch: int):
        """Update current epoch"""
        self.current_epoch = epoch

    def add_train_loss(self, loss: float):
        """Add training loss for current epoch"""
        self.train_losses.append(loss)

    def add_val_loss(self, loss: float):
        """Add validation loss for current epoch"""
        self.val_losses.append(loss)

    def add_train_accuracy(self, accuracy: float):
        """Add training accuracy for current epoch"""
        self.train_accuracies.append(accuracy)

    def add_val_accuracy(self, accuracy: float):
        """Add validation accuracy for current epoch"""
        self.val_accuracies.append(accuracy)

    def add_epoch_complete(self):
        """Mark epoch as complete"""
        if self.current_epoch not in self.epochs:
            self.epochs.append(self.current_epoch)

    def plot_losses(self, save_path: Optional[str] = None, show_plot: bool = True):
        """Plot training and validation losses"""
        plt.figure(figsize=(12, 4))

        # Plot losses
        plt.subplot(1, 2, 1)
        if self.train_losses:
            # Align losses with epochs
            train_epochs = self.epochs[:len(self.train_losses)]
            plt.plot(train_epochs, self.train_losses, 'b-', label='Train Loss', linewidth=2)

        if self.val_losses:
            val_epochs = self.epochs[:len(self.val_losses)]
            plt.plot(val_epochs, self.val_losses, 'r-', label='Val Loss', linewidth=2)

        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training and Validation Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Plot accuracies
        plt.subplot(1, 2, 2)
        if self.train_accuracies:
            train_acc_epochs = self.epochs[:len(self.train_accuracies)]
            plt.plot(train_acc_epochs, self.train_accuracies, 'b-', label='Train Acc', linewidth=2)

        if self.val_accuracies:
            val_acc_epochs = self.epochs[:len(self.val_accuracies)]
            plt.plot(val_acc_epochs, self.val_accuracies, 'r-', label='Val Acc', linewidth=2)

        plt.xlabel('Epoch')
        plt.ylabel('Accuracy (%)')
        plt.title('Training and Validation Accuracy')
        plt.legend()
        plt.grid(True, alpha=0.3)

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Plot saved to: {save_path}")

        if show_plot:
            plt.show()
        else:
            plt.close()

    def save_metrics(self, save_path: str):
        """Save metrics to a text file"""
        with open(save_path, 'w') as f:
            f.write("Epoch,Train_Loss,Val_Loss,Train_Acc,Val_Acc\n")
            max_len = max(len(self.epochs), len(self.train_losses), len(self.val_losses),
                         len(self.train_accuracies), len(self.val_accuracies))

            for i in range(max_len):
                epoch = self.epochs[i] if i < len(self.epochs) else ""
                train_loss = self.train_losses[i] if i < len(self.train_losses) else ""
                val_loss = self.val_losses[i] if i < len(self.val_losses) else ""
                train_acc = self.train_accuracies[i] if i < len(self.train_accuracies) else ""
                val_acc = self.val_accuracies[i] if i < len(self.val_accuracies) else ""

                f.write(f"{epoch},{train_loss},{val_loss},{train_acc},{val_acc}\n")

        print(f"Metrics saved to: {save_path}")

## TRAIN & VALIDATE


In [12]:
def train_model(config_path: str):
    """
    Train the SVTR model
    """
    os.chdir('/content/PaddleOCR')

    cmd = ["python3", "tools/train.py", "-c", config_path]

    try:
        print(f"Starting training...")
        print(f"Command: {' '.join(cmd)}")

        result = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            universal_newlines=True
        )

        best_acc = 0.0
        current_epoch = 0
        last_train_acc = 0.0
        last_val_acc = 0.0
        current_train_loss = None
        current_val_loss = None

        # Track 500-step averages
        train_acc_window = []
        train_loss_window = []
        step_count = 0
        last_global_step = 0
        current_global_step = 0

        # Read and print the output line by line
        for line in iter(result.stdout.readline, ''):
            print(line, end='')

            # Parse epoch - fix pattern for your logs
            epoch_match = re.search(r'epoch:\s*\[(\d+)/(\d+)\]', line)
            if epoch_match:
                current_epoch = int(epoch_match.group(1))

            # Parse global step
            global_step_match = re.search(r'global_step:\s*(\d+)', line)
            if global_step_match:
                current_global_step = int(global_step_match.group(1))

            # Parse training loss - be more specific to avoid conflicts
            train_loss_match = re.search(r'loss:\s*([0-9.]+)', line)
            if train_loss_match and 'eval' not in line.lower():
                current_train_loss = float(train_loss_match.group(1))
                train_loss_window.append(current_train_loss)
                print(f"[MONITOR] Step {current_global_step} | Train Loss: {current_train_loss:.4f}")

            # Parse training accuracy - fix for decimal format
            train_acc_match = re.search(r'acc:\s*([0-9.]+)', line)
            if train_acc_match and 'eval' not in line.lower():
                last_train_acc = float(train_acc_match.group(1)) * 100  # Convert to percentage
                train_acc_window.append(last_train_acc)
                print(f"[MONITOR] Step {current_global_step} | Train Acc: {last_train_acc:.2f}%")

            # Check if we've completed a 500-step window
            if current_global_step > 0 and current_global_step % 500 == 0 and current_global_step != last_global_step:
                if train_acc_window and train_loss_window:
                    avg_train_acc = sum(train_acc_window) / len(train_acc_window)
                    avg_train_loss = sum(train_loss_window) / len(train_loss_window)

                    print(f"\n[500-STEP SUMMARY] Steps {current_global_step-499} to {current_global_step}:")
                    print(f"  Average Train Acc: {avg_train_acc:.2f}%")
                    print(f"  Average Train Loss: {avg_train_loss:.4f}")
                    print(f"  Samples in window: {len(train_acc_window)}")
                    print("-" * 60)

                    # Clear windows for next 500 steps
                    train_acc_window = []
                    train_loss_window = []

                last_global_step = current_global_step

            # Parse validation accuracy - fix for your log format
            val_acc_match = re.search(r'cur metric, acc:\s*([0-9.]+)', line)
            if val_acc_match:
                last_val_acc = float(val_acc_match.group(1)) * 100  # Convert to percentage

                print(f"\n[EVAL RESULT] Epoch {current_epoch}:")
                print(f"  Validation Acc: {last_val_acc:.2f}%")

                if avg_train_acc > 0:
                    acc_gap = abs(avg_train_acc - last_val_acc)
                    print(f"  Train Acc (500-step avg): {avg_train_acc:.2f}%")
                    print(f"  Gap: {acc_gap:.1f}% {'⚠️ LARGE GAP!' if acc_gap > 8 else '⚠️ MODERATE GAP' if acc_gap > 5 else '✓ Healthy gap'}")
                else:
                    print(f"  Gap: N/A (no 500-step average yet)")

                print("=" * 50)

        result.wait()

        if result.returncode == 0:
            print("Training completed successfully!")
            return True
        else:
            print(f"Training failed with return code: {result.returncode}")
            return False
    except subprocess.CalledProcessError as e:
        print(f"Training failed with error: {e}")
        return False
    finally:
        os.chdir('/content')

def evaluate_model(config_path: str, model_path: str):
    """Fixed evaluation with better parsing"""
    os.chdir('/content/PaddleOCR')

    cmd = [
        "python3", "tools/eval.py",
        "-c", config_path,
        "-o", f"Global.checkpoints={model_path}"
    ]

    print(f"Starting evaluation with command: {' '.join(cmd)}")

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, check=False)
        print("Evaluation output:")
        print(result.stdout)

        if result.stderr:
            print("Evaluation stderr:")
            print(result.stderr)

        # Better accuracy parsing
        lines = result.stdout.split('\n')
        accuracy = 0.0

        for line in lines:
            # Look for the final metric evaluation line
            if 'metric eval' in line.lower():
                continue
            # Parse accuracy from the metric output
            if 'acc:' in line and not 'best_epoch' in line:
                acc_match = re.search(r'acc:\s*([0-9.]+)', line)
                if acc_match:
                    acc_value = float(acc_match.group(1))
                    if acc_value > accuracy:
                        accuracy = acc_value

        print(f"Parsed accuracy: {accuracy:.4f}")
        return accuracy

    except Exception as e:
        print(f"Evaluation error: {e}")
        return 0.0
    finally:
        os.chdir('/content')

## DEBUG

In [10]:
def quick_fix_and_retry_training(config_path: str):

    print("=== QUICK FIX: Replacing MultiLabelEncode with CTCLabelEncode ===")

    # Read and fix config
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    # Fix training transforms
    for transform in config['Train']['dataset']['transforms']:
        if 'MultiLabelEncode' in transform:
            transform.clear()
            transform['CTCLabelEncode'] = {}

    # Fix eval transforms
    for transform in config['Eval']['dataset']['transforms']:
        if 'MultiLabelEncode' in transform:
            transform.clear()
            transform['CTCLabelEncode'] = {}

    # Remove problematic augmentations
    config['Train']['dataset']['transforms'] = [
        t for t in config['Train']['dataset']['transforms']
        if not any(key in ['RecConAug', 'RecAug'] for key in t.keys())
    ]

    # Save fixed config
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

    print("Config fixed! Retrying training...")
    return train_model(config_path)


In [11]:
def complete_cuda_reset():
    """Complete CUDA environment reset and reinstallation"""
    import os
    import subprocess
    import sys
    import time

    print("=== CUDA Environment Diagnosis and Reset ===")

    # 1. Check current CUDA status
    print("\n1. Current CUDA Status:")
    os.system('nvidia-smi')
    os.system('nvcc --version')

    # 2. Kill all CUDA processes
    print("\n2. Killing all CUDA processes...")
    os.system('pkill -f python')
    os.system('pkill -f cuda')
    time.sleep(3)

    # 3. Reset GPU completely
    print("\n3. Resetting GPU...")
    os.system('nvidia-smi --gpu-reset')
    os.system('nvidia-smi --reset-gpus')
    time.sleep(5)

    # 4. Clear all Python CUDA-related modules
    print("\n4. Clearing Python modules...")
    modules_to_remove = [
        'paddle', 'paddlepaddle', 'paddlepaddle-gpu',
        'torch', 'tensorflow', 'cupy'
    ]

    for module in list(sys.modules.keys()):
        if any(mod in module.lower() for mod in modules_to_remove):
            print(f"Removing module: {module}")
            del sys.modules[module]

    # 5. Reinstall PaddlePaddle with specific CUDA version
    print("\n5. Reinstalling PaddlePaddle...")
    os.system('pip uninstall -y paddlepaddle paddlepaddle-gpu')
    time.sleep(2)

    # Install specific PaddlePaddle version known to work
    install_cmd = 'pip install paddlepaddle-gpu==2.5.1.post117 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html'
    os.system(install_cmd)

    print("\n6. CUDA Reset Complete. Please restart your runtime and run the training script.")

# SOLUTION 2: Alternative PaddlePaddle Installation
# If above doesn't work, try this specific version that's more stable in Colab

def install_stable_paddle():
    """Install a more stable PaddlePaddle version"""
    import os
    print("Installing stable PaddlePaddle version...")

    # Uninstall current version
    os.system('pip uninstall -y paddlepaddle paddlepaddle-gpu')

    # Install older stable version
    os.system('pip install paddlepaddle-gpu==2.4.2.post117 -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html')


# Historical runs

In [ ]:
if __name__ == '__main__':
    # 1. Mount Google Drive and set up paths.
    # Run the reset
    from google.colab import drive
    drive.mount('/content/drive')

    # Define the source paths for the SROIE dataset
    train_source_img_path = '/content/drive/MyDrive/SROIE2019/train/img'
    train_source_box_path = '/content/drive/MyDrive/SROIE2019/train/box'
    val_test_source_img_path = '/content/drive/MyDrive/SROIE2019/test/img'
    val_test_source_box_path = '/content/drive/MyDrive/SROIE2019/test/box'
    local_data_path = '/content/svtr_data_temp'

    # Download pretrained model
    print("--- Downloading Pretrained Model ---")
    pretrained_model_path = download_pretrained_model()

    if pretrained_model_path:
        print(f"Pretrained model ready: {pretrained_model_path}")

        # Get character dictionary
        char_dict_path = get_character_dict_path()
        print(f"Character dictionary: {char_dict_path}")

        # Create training configuration
        work_dir = '/content/drive/MyDrive/svtr_training'
        best_curr = '/content/drive/MyDrive/best_accuracy'

        config_path = create_svtr_config(
            work_dir=work_dir,
            data_dir=local_data_path,
            pretrained_model_path=pretrained_model_path,
            char_dict_path=char_dict_path,
            checkpoint_path=None
        )

        print(f"Configuration created: {config_path}")

        print("--- Starting Training with Debugging ---")
        success = quick_fix_and_retry_training(config_path)
        if success:
          # Find the best model
            model_dir = f"{work_dir}/output/rec_svtr"
            best_model_path = os.path.join(model_dir, "best_accuracy.pdparams")
            if os.path.exists(best_model_path):
                print(f"Best model found: {best_model_path}")

                # Evaluate on validation set
                print("--- Evaluating on Validation Set ---")
                val_accuracy = evaluate_model(config_path, best_model_path)
                print(f"Validation accuracy: {val_accuracy:.4f}")

                # Evaluate on test set
                print("--- Final Evaluation on Test Set ---")
                # Create test-specific config
                test_accuracy = evaluate_model(config_path, best_model_path)
                print(f"Final test accuracy: {test_accuracy:.4f}")

                print("\n=== FINAL RESULTS ===")
                print(f"Best model: {best_model_path}")
                print(f"Validation accuracy: {val_accuracy:.4f}")
                print(f"Test accuracy: {test_accuracy:.4f}")
            else:
                print(f"Best model not found at {best_model_path}")
                # List available models
                model_files = glob(os.path.join(model_dir, "*.pdparams"))
                print(f"Available models: {model_files}")
        else:
            print("Training failed!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Downloading Pretrained Model ---
Extracting to /content/PaddleOCR/pretrain_models/en_ppocr_v3_rec/...


/tmp/ipython-input-1257119269.py:84: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(model_config['extract_dir'])


✅ Successfully downloaded model to /content/PaddleOCR/pretrain_models/en_ppocr_v3_rec/en_PP-OCRv3_rec_train/best_accuracy.pdparams
Pretrained model ready: /content/PaddleOCR/pretrain_models/en_ppocr_v3_rec/en_PP-OCRv3_rec_train/best_accuracy.pdparams
Character dictionary: /content/PaddleOCR/ppocr/utils/en_dict.txt
Configuration created: /content/drive/MyDrive/svtr_training/config.yml
--- Starting Training with Debugging ---
=== QUICK FIX: Replacing MultiLabelEncode with CTCLabelEncode ===
Config fixed! Retrying training...
Starting training...
Command: python3 tools/train.py -c /content/drive/MyDrive/svtr_training/config.yml
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:717: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
Tr

## USAGE
###1st run

In [ ]:
if __name__ == '__main__':
    # 1. Mount Google Drive and set up paths.
    from google.colab import drive
    drive.mount('/content/drive')

    # Define the source paths for the SROIE dataset
    train_source_img_path = '/content/drive/MyDrive/SROIE2019/train/img'
    train_source_box_path = '/content/drive/MyDrive/SROIE2019/train/box'
    val_test_source_img_path = '/content/drive/MyDrive/SROIE2019/test/img'
    val_test_source_box_path = '/content/drive/MyDrive/SROIE2019/test/box'

    # 2. Combine and prepare the data locally for better performance and reliability.
    local_data_path = '/content/svtr_data_temp'
    # Ensure the local data directory and its subdirectories exist
    if os.path.exists(local_data_path):
        shutil.rmtree(local_data_path)  # Remove old local data
    os.makedirs(os.path.join(local_data_path, 'train/img'), exist_ok=True)
    os.makedirs(os.path.join(local_data_path, 'val/img'), exist_ok=True)
    os.makedirs(os.path.join(local_data_path, 'test/img'), exist_ok=True)

    TRAIN_SET_SIZE = 250
    VALIDATION_SET_SIZE = 50
    TEST_SET_SIZE = 0

    # Create a single list of all image files and a combined box directory
    all_img_files = glob(os.path.join(train_source_img_path, '*.jpg')) + glob(os.path.join(val_test_source_img_path, '*.jpg'))

    if len(all_img_files) >= TRAIN_SET_SIZE + VALIDATION_SET_SIZE + TEST_SET_SIZE:
        random.seed(42)
        random.shuffle(all_img_files)

        train_files = all_img_files[:TRAIN_SET_SIZE]
        val_files = all_img_files[TRAIN_SET_SIZE:TRAIN_SET_SIZE + VALIDATION_SET_SIZE]
        test_files = all_img_files[TRAIN_SET_SIZE + VALIDATION_SET_SIZE:
                                TRAIN_SET_SIZE + VALIDATION_SET_SIZE + TEST_SET_SIZE]

        # Process with improved function
        source_box_paths = [train_source_box_path, val_test_source_box_path]

        print("\n--- Processing Training Set ---")
        train_count = data_loader(
            train_files, source_box_paths,
            os.path.join(local_data_path, 'train/img'),
            os.path.join(local_data_path, 'train/train_gt.txt'),
            enable_augmentation = True,
        )

        print("\n--- Processing Validation Set ---")
        val_count = data_loader(
            val_files, source_box_paths,
            os.path.join(local_data_path, 'val/img'),
            os.path.join(local_data_path, 'val/val_gt.txt'),
            enable_augmentation = False,
            augment_multiplier=0
        )

        print(f"\nDataset prepared:")
        print(f"  Training: {train_count} samples")
        print(f"  Validation: {val_count} samples")
        print(f"  Test: {test_count} samples")

        if True: # train_count > 0 and val_count > 0 and test_count > 0
            print("\n--- Checking Data Quality ---")
            #check_data_quality(local_data_path)

            # Download pretrained model
            print("--- Downloading Pretrained Model ---")
            pretrained_model_path = download_pretrained_model()

            if pretrained_model_path and os.path.exists(pretrained_model_path):
                print(f"Pretrained model ready: {pretrained_model_path}")

                # Get character dictionary
                char_dict_path = get_character_dict_path()
                print(f"Character dictionary: {char_dict_path}")

                # Create training configuration
                work_dir = '/content/drive/MyDrive/svtr_training'
                best_curr = '/content/drive/MyDrive/best_accuracy'

                config_path = create_svtr_config(
                    work_dir=work_dir,
                    data_dir=local_data_path,
                    pretrained_model_path=pretrained_model_path, # set to None
                    char_dict_path=char_dict_path,
                    checkpoint_path=None #set to best_curr when continue to train
                )

                print(f"Configuration created: {config_path}")

                print("--- Starting Training with Debugging ---")
                success = quick_fix_and_retry_training(config_path)
                if success:
                    # Find the best model
                    model_dir = f"{work_dir}/output/rec_svtr"
                    best_model_path = os.path.join(model_dir, "best_accuracy.pdparams")

                    if os.path.exists(best_model_path):
                        print(f"Best model found: {best_model_path}")

                        # Evaluate on validation set
                        print("--- Evaluating on Validation Set ---")
                        val_accuracy = evaluate_model(config_path, best_model_path)
                        print(f"Validation accuracy: {val_accuracy:.4f}")

                        # Evaluate on test set
                        print("--- Final Evaluation on Test Set ---")
                        # Create test-specific config
                        test_accuracy = evaluate_model(config_path, best_model_path)
                        print(f"Final test accuracy: {test_accuracy:.4f}")

                        print("\n=== FINAL RESULTS ===")
                        print(f"Best model: {best_model_path}")
                        print(f"Validation accuracy: {val_accuracy:.4f}")
                        print(f"Test accuracy: {test_accuracy:.4f}")
                    else:
                        print(f"Best model not found at {best_model_path}")
                        # List available models
                        model_files = glob(os.path.join(model_dir, "*.pdparams"))
                        print(f"Available models: {model_files}")
                else:
                    print("Training failed!")
            else:
                print("Failed to download pretrained model!")
        else:
            print("No valid training data found!")
    else:
        print(f"Not enough images. Found {len(all_img_files)}, need {TRAIN_SET_SIZE + VALIDATION_SET_SIZE + TEST_SET_SIZE}")

Mounted at /content/drive

--- Processing Training Set ---


/tmp/ipython-input-2534086894.py:64: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=10, p=0.4),
Processing img data: 100%|██████████| 250/250 [03:02<00:00,  1.37it/s]


Processing stats for img:
  Total text regions processed: 13056
  Valid crops created: 13055
  Total augmented crops: 52220
  Images without box files: 0
  Invalid/skipped regions: 2
  Success rate: 100.0%

--- Processing Validation Set ---


Processing img data: 100%|██████████| 50/50 [00:21<00:00,  2.33it/s]


Processing stats for img:
  Total text regions processed: 2910
  Valid crops created: 2909
  Total augmented crops: 2909
  Images without box files: 0
  Invalid/skipped regions: 1
  Success rate: 100.0%

--- Processing Test Set ---


Processing img data: 0it [00:00, ?it/s]


Processing stats for img:
  Total text regions processed: 0
  Valid crops created: 0
  Total augmented crops: 0
  Images without box files: 0
  Invalid/skipped regions: 0
  Success rate: 0.0%

Dataset prepared:
  Training: 13055 samples
  Validation: 2909 samples
  Test: 0 samples

--- Checking Data Quality ---
--- Downloading Pretrained Model ---
Extracting to /content/PaddleOCR/pretrain_models/en_ppocr_v3_rec/...


/tmp/ipython-input-1888511781.py:50: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(model_config['extract_dir'])


✅ Successfully downloaded model to /content/PaddleOCR/pretrain_models/en_ppocr_v3_rec/en_PP-OCRv3_rec_train/best_accuracy.pdparams
Pretrained model ready: /content/PaddleOCR/pretrain_models/en_ppocr_v3_rec/en_PP-OCRv3_rec_train/best_accuracy.pdparams
Character dictionary: /content/PaddleOCR/ppocr/utils/en_dict.txt
Configuration created: /content/drive/MyDrive/svtr_training/config.yml
--- Starting Training with Debugging ---
=== QUICK FIX: Replacing MultiLabelEncode with CTCLabelEncode ===
Config fixed! Retrying training...
Starting training...
Command: python3 tools/train.py -c /content/drive/MyDrive/svtr_training/config.yml
[2025/09/07 06:56:33] ppocr INFO: Architecture : 
[2025/09/07 06:56:33] ppocr INFO:     Backbone : 
[2025/09/07 06:56:33] ppocr INFO:         depth : [3, 6, 3]
[2025/09/07 06:56:33] ppocr INFO:         drop_path : 0.1
[2025/09/07 06:56:33] ppocr INFO:         drop_rate : 0.1
[2025/09/07 06:56:33] ppocr INFO:         embed_dim : [64, 128, 256]
[2025/09/07 06:56:33] 

## USAGE
###2nd run
drop_rate 0.1 -> 0.2

drop_path 0.1 -> 0.2

weight_decay 0.1 -> 0.2

batch_size -> 32

In [ ]:
if __name__ == '__main__':
    # 1. Mount Google Drive and set up paths.
    # Run the reset
    from google.colab import drive
    drive.mount('/content/drive')

    # Define the source paths for the SROIE dataset
    train_source_img_path = '/content/drive/MyDrive/SROIE2019/train/img'
    train_source_box_path = '/content/drive/MyDrive/SROIE2019/train/box'
    val_test_source_img_path = '/content/drive/MyDrive/SROIE2019/test/img'
    val_test_source_box_path = '/content/drive/MyDrive/SROIE2019/test/box'
    local_data_path = '/content/svtr_data_temp'

    # Download pretrained model
    print("--- Downloading Pretrained Model ---")
    pretrained_model_path = download_pretrained_model()

    if pretrained_model_path:
        print(f"Pretrained model ready: {pretrained_model_path}")

        # Get character dictionary
        char_dict_path = get_character_dict_path()
        print(f"Character dictionary: {char_dict_path}")

        # Create training configuration
        work_dir = '/content/drive/MyDrive/svtr_training'
        best_curr = '/content/drive/MyDrive/best_accuracy'

        config_path = create_svtr_config(
            work_dir=work_dir,
            data_dir=local_data_path,
            pretrained_model_path=None,
            char_dict_path=char_dict_path,
            checkpoint_path=best_curr
        )

        print(f"Configuration created: {config_path}")

        print("--- Starting Training with Debugging ---")
        success = quick_fix_and_retry_training(config_path)
        if success:
          # Find the best model
            model_dir = f"{work_dir}/output/rec_svtr"
            best_model_path = os.path.join(model_dir, "best_accuracy.pdparams")
            if os.path.exists(best_model_path):
                print(f"Best model found: {best_model_path}")

                # Evaluate on validation set
                print("--- Evaluating on Validation Set ---")
                val_accuracy = evaluate_model(config_path, best_model_path)
                print(f"Validation accuracy: {val_accuracy:.4f}")

                # Evaluate on test set
                print("--- Final Evaluation on Test Set ---")
                # Create test-specific config
                test_accuracy = evaluate_model(config_path, best_model_path)
                print(f"Final test accuracy: {test_accuracy:.4f}")

                print("\n=== FINAL RESULTS ===")
                print(f"Best model: {best_model_path}")
                print(f"Validation accuracy: {val_accuracy:.4f}")
                print(f"Test accuracy: {test_accuracy:.4f}")
            else:
                print(f"Best model not found at {best_model_path}")
                # List available models
                model_files = glob(os.path.join(model_dir, "*.pdparams"))
                print(f"Available models: {model_files}")
        else:
            print("Training failed!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Downloading Pretrained Model ---
Pretrained model already exists at: /content/PaddleOCR/pretrain_models/en_ppocr_v3_rec/en_PP-OCRv3_rec_train/best_accuracy.pdparams
Pretrained model ready: /content/PaddleOCR/pretrain_models/en_ppocr_v3_rec/en_PP-OCRv3_rec_train/best_accuracy.pdparams
Character dictionary: /content/PaddleOCR/ppocr/utils/en_dict.txt
Configuration created: /content/drive/MyDrive/svtr_training/config.yml
--- Starting Training with Debugging ---
=== QUICK FIX: Replacing MultiLabelEncode with CTCLabelEncode ===
Config fixed! Retrying training...
Starting training...
Command: python3 tools/train.py -c /content/drive/MyDrive/svtr_training/config.yml
[2025/09/07 08:58:57] ppocr INFO: Architecture : 
[2025/09/07 08:58:57] ppocr INFO:     Backbone : 
[2025/09/07 08:58:57] ppocr INFO:         depth : [3, 6, 3]
[2025/09/07 08:58:57] ppocr INFO:       

KeyboardInterrupt: 

## USAGE
###3rd run
drop_rate 0.2 -> 0.1

drop_path 0.2 -> 0.1

weight_decay 0.2 -> 0.1

batch_size 32 -> 64

###4th run
drop_rate 0.05

drop_path 0.05

weight_decay 0.05

batch_size 80 (128 out mem)


In [ ]:
if __name__ == '__main__':
    # 1. Mount Google Drive and set up paths.
    from google.colab import drive
    drive.mount('/content/drive')

    # Define the source paths for the SROIE dataset
    train_source_img_path = '/content/drive/MyDrive/SROIE2019/train/img'
    train_source_box_path = '/content/drive/MyDrive/SROIE2019/train/box'
    val_test_source_img_path = '/content/drive/MyDrive/SROIE2019/test/img'
    val_test_source_box_path = '/content/drive/MyDrive/SROIE2019/test/box'

    # 2. Combine and prepare the data locally for better performance and reliability.
    local_data_path = '/content/svtr_data_temp'
    # Ensure the local data directory and its subdirectories exist
    if os.path.exists(local_data_path):
        shutil.rmtree(local_data_path)  # Remove old local data
    os.makedirs(os.path.join(local_data_path, 'train/img'), exist_ok=True)
    os.makedirs(os.path.join(local_data_path, 'val/img'), exist_ok=True)
    os.makedirs(os.path.join(local_data_path, 'test/img'), exist_ok=True)

    TRAIN_SET_SIZE = 250
    VALIDATION_SET_SIZE = 50
    TEST_SET_SIZE = 0

    # Create a single list of all image files and a combined box directory
    all_img_files = glob(os.path.join(train_source_img_path, '*.jpg')) + glob(os.path.join(val_test_source_img_path, '*.jpg'))

    if len(all_img_files) >= TRAIN_SET_SIZE + VALIDATION_SET_SIZE + TEST_SET_SIZE:
        random.seed(42)
        random.shuffle(all_img_files)

        train_files = all_img_files[:TRAIN_SET_SIZE]
        val_files = all_img_files[TRAIN_SET_SIZE:TRAIN_SET_SIZE + VALIDATION_SET_SIZE]
        test_files = all_img_files[TRAIN_SET_SIZE + VALIDATION_SET_SIZE:
                                TRAIN_SET_SIZE + VALIDATION_SET_SIZE + TEST_SET_SIZE]

        # Process with improved function
        source_box_paths = [train_source_box_path, val_test_source_box_path]

        print("\n--- Processing Training Set ---")
        train_count = data_loader(
            train_files, source_box_paths,
            os.path.join(local_data_path, 'train/img'),
            os.path.join(local_data_path, 'train/train_gt.txt'),
            enable_augmentation = True,
        )

        print("\n--- Processing Validation Set ---")
        val_count = data_loader(
            val_files, source_box_paths,
            os.path.join(local_data_path, 'val/img'),
            os.path.join(local_data_path, 'val/val_gt.txt'),
            enable_augmentation = False,
            augment_multiplier=0
        )

        print(f"\nDataset prepared:")
        print(f"  Training: {train_count} samples")
        print(f"  Validation: {val_count} samples")

        if True: # train_count > 0 and val_count > 0 and test_count > 0
            print("\n--- Checking Data Quality ---")
            #check_data_quality(local_data_path)

            # Download pretrained model
            print("--- Downloading Pretrained Model ---")
            pretrained_model_path = download_pretrained_model()

            if pretrained_model_path and os.path.exists(pretrained_model_path):
                print(f"Pretrained model ready: {pretrained_model_path}")

                # Get character dictionary
                char_dict_path = get_character_dict_path()
                print(f"Character dictionary: {char_dict_path}")

                # Create training configuration
                work_dir = '/content/drive/MyDrive/svtr_training'
                best_curr = '/content/drive/MyDrive/best_accuracy'

                config_path = create_svtr_config(
                    work_dir=work_dir,
                    data_dir=local_data_path,
                    pretrained_model_path=None, # set to None
                    char_dict_path=char_dict_path,
                    checkpoint_path=best_curr #set to best_curr when continue to train
                )

                print(f"Configuration created: {config_path}")

                print("--- Starting Training with Debugging ---")
                success = quick_fix_and_retry_training(config_path)
                if success:
                    # Find the best model
                    model_dir = f"{work_dir}/output/rec_svtr"
                    best_model_path = os.path.join(model_dir, "best_accuracy.pdparams")

                    if os.path.exists(best_model_path):
                        print(f"Best model found: {best_model_path}")

                        # Evaluate on validation set
                        print("--- Evaluating on Validation Set ---")
                        val_accuracy = evaluate_model(config_path, best_model_path)
                        print(f"Validation accuracy: {val_accuracy:.4f}")

                        # Evaluate on test set
                        print("--- Final Evaluation on Test Set ---")
                        # Create test-specific config
                        test_accuracy = evaluate_model(config_path, best_model_path)
                        print(f"Final test accuracy: {test_accuracy:.4f}")

                        print("\n=== FINAL RESULTS ===")
                        print(f"Best model: {best_model_path}")
                        print(f"Validation accuracy: {val_accuracy:.4f}")
                        print(f"Test accuracy: {test_accuracy:.4f}")
                    else:
                        print(f"Best model not found at {best_model_path}")
                        # List available models
                        model_files = glob(os.path.join(model_dir, "*.pdparams"))
                        print(f"Available models: {model_files}")
                else:
                    print("Training failed!")
            else:
                print("Failed to download pretrained model!")
        else:
            print("No valid training data found!")
    else:
        print(f"Not enough images. Found {len(all_img_files)}, need {TRAIN_SET_SIZE + VALIDATION_SET_SIZE + TEST_SET_SIZE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Downloading Pretrained Model ---
Pretrained model already exists at: /content/PaddleOCR/pretrain_models/en_ppocr_v3_rec/en_PP-OCRv3_rec_train/best_accuracy.pdparams
Pretrained model ready: /content/PaddleOCR/pretrain_models/en_ppocr_v3_rec/en_PP-OCRv3_rec_train/best_accuracy.pdparams
Character dictionary: /content/PaddleOCR/ppocr/utils/en_dict.txt
Configuration created: /content/drive/MyDrive/svtr_training/config.yml
--- Starting Training with Debugging ---
=== QUICK FIX: Replacing MultiLabelEncode with CTCLabelEncode ===
Config fixed! Retrying training...
Starting training...
Command: python3 tools/train.py -c /content/drive/MyDrive/svtr_training/config.yml
[2025/09/07 12:22:13] ppocr INFO: Architecture : 
[2025/09/07 12:22:13] ppocr INFO:     Backbone : 
[2025/09/07 12:22:13] ppocr INFO:         depth : [3, 6, 3]
[2025/09/07 12:22:13] ppocr INFO:       

In [17]:
local_data_path = '/content/svtr_data_temp'
work_dir = '/content/drive/MyDrive/svtr_training'
best_curr = '/content/drive/MyDrive/best_accuracy'
char_dict_path = '/content/PaddleOCR/ppocr/utils/en_dict.txt'
config_path = create_svtr_config(
                    work_dir=work_dir,
                    data_dir=local_data_path,
                    pretrained_model_path=None, # set to None
                    char_dict_path=char_dict_path,
                    checkpoint_path=None #set to best_curr when continue to train
                )
success = train_model(config_path)

Starting training...
Command: python3 tools/train.py -c /content/drive/MyDrive/svtr_training/config.yml
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:717: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Skipping import of the encryption module.
Traceback (most recent call last):
  File "/content/PaddleOCR/tools/train.py", line 269, in <module>
    config, device, logger, vdl_writer = program.preprocess(is_train=True)
                                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/PaddleOCR/tools/program.py", line 900, in preprocess
    device = "gpu:{}".format(dist.ParallelEnv().dev_id) if use_gpu else "cpu"
                             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/paddle/distributed/parallel.p

# Current run

In [ ]:
if __name__ == '__main__':
    # 1. Mount Google Drive and set up paths.
    from google.colab import drive
    drive.mount('/content/drive')

    # Define the source paths for the SROIE dataset
    train_source_img_path = '/content/drive/MyDrive/SROIE2019/train/img'
    train_source_box_path = '/content/drive/MyDrive/SROIE2019/train/box'
    val_test_source_img_path = '/content/drive/MyDrive/SROIE2019/test/img'
    val_test_source_box_path = '/content/drive/MyDrive/SROIE2019/test/box'

    # 2. Combine and prepare the data locally for better performance and reliability.
    local_data_path = '/content/svtr_data_temp'
    # Ensure the local data directory and its subdirectories exist
    if os.path.exists(local_data_path):
        shutil.rmtree(local_data_path)  # Remove old local data
    os.makedirs(os.path.join(local_data_path, 'train/img'), exist_ok=True)
    os.makedirs(os.path.join(local_data_path, 'val/img'), exist_ok=True)
    os.makedirs(os.path.join(local_data_path, 'test/img'), exist_ok=True)

    TRAIN_SET_SIZE = 250
    VALIDATION_SET_SIZE = 50
    TEST_SET_SIZE = 0

    # Create a single list of all image files and a combined box directory
    all_img_files = glob(os.path.join(train_source_img_path, '*.jpg')) + glob(os.path.join(val_test_source_img_path, '*.jpg'))

    if len(all_img_files) >= TRAIN_SET_SIZE + VALIDATION_SET_SIZE + TEST_SET_SIZE:
        random.seed(42)
        random.shuffle(all_img_files)

        train_files = all_img_files[:TRAIN_SET_SIZE]
        val_files = all_img_files[TRAIN_SET_SIZE:TRAIN_SET_SIZE + VALIDATION_SET_SIZE]
        test_files = all_img_files[TRAIN_SET_SIZE + VALIDATION_SET_SIZE:
                                TRAIN_SET_SIZE + VALIDATION_SET_SIZE + TEST_SET_SIZE]

        # Process with improved function
        source_box_paths = [train_source_box_path, val_test_source_box_path]

        print("\n--- Processing Training Set ---")
        train_count = data_loader(
            train_files, source_box_paths,
            os.path.join(local_data_path, 'train/img'),
            os.path.join(local_data_path, 'train/train_gt.txt'),
            enable_augmentation = True,
        )

        print("\n--- Processing Validation Set ---")
        val_count = data_loader(
            val_files, source_box_paths,
            os.path.join(local_data_path, 'val/img'),
            os.path.join(local_data_path, 'val/val_gt.txt'),
            enable_augmentation = False,
            augment_multiplier=0
        )

        print(f"\nDataset prepared:")
        print(f"  Training: {train_count} samples")
        print(f"  Validation: {val_count} samples")

        if True: # train_count > 0 and val_count > 0 and test_count > 0
            print("\n--- Checking Data Quality ---")
            #check_data_quality(local_data_path)

            # Download pretrained model
            print("--- Downloading Pretrained Model ---")
            pretrained_model_path = download_pretrained_model()

            if pretrained_model_path and os.path.exists(pretrained_model_path):
                print(f"Pretrained model ready: {pretrained_model_path}")

                # Get character dictionary
                char_dict_path = get_character_dict_path()
                print(f"Character dictionary: {char_dict_path}")

                # Create training configuration
                work_dir = '/content/drive/MyDrive/svtr_training'
                best_curr = '/content/drive/MyDrive/best_accuracy'

                config_path = create_svtr_config(
                    work_dir=work_dir,
                    data_dir=local_data_path,
                    pretrained_model_path=None, # set to None
                    char_dict_path=char_dict_path,
                    checkpoint_path=best_curr #set to best_curr when continue to train
                )

                print(f"Configuration created: {config_path}")

                print("--- Starting Training with Debugging ---")
                success = quick_fix_and_retry_training(config_path)
                if success:
                    # Find the best model
                    model_dir = f"{work_dir}/output/rec_svtr"
                    best_model_path = os.path.join(model_dir, "best_accuracy.pdparams")

                    if os.path.exists(best_model_path):
                        print(f"Best model found: {best_model_path}")

                        # Evaluate on validation set
                        print("--- Evaluating on Validation Set ---")
                        val_accuracy = evaluate_model(config_path, best_model_path)
                        print(f"Validation accuracy: {val_accuracy:.4f}")

                        # Evaluate on test set
                        print("--- Final Evaluation on Test Set ---")
                        # Create test-specific config
                        test_accuracy = evaluate_model(config_path, best_model_path)
                        print(f"Final test accuracy: {test_accuracy:.4f}")

                        print("\n=== FINAL RESULTS ===")
                        print(f"Best model: {best_model_path}")
                        print(f"Validation accuracy: {val_accuracy:.4f}")
                        print(f"Test accuracy: {test_accuracy:.4f}")
                    else:
                        print(f"Best model not found at {best_model_path}")
                        # List available models
                        model_files = glob(os.path.join(model_dir, "*.pdparams"))
                        print(f"Available models: {model_files}")
                else:
                    print("Training failed!")
            else:
                print("Failed to download pretrained model!")
        else:
            print("No valid training data found!")
    else:
        print(f"Not enough images. Found {len(all_img_files)}, need {TRAIN_SET_SIZE + VALIDATION_SET_SIZE + TEST_SET_SIZE}")

Mounted at /content/drive


/tmp/ipython-input-2534086894.py:64: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=10, p=0.4),



--- Processing Training Set ---


Processing img data: 100%|██████████| 250/250 [03:06<00:00,  1.34it/s]


Processing stats for img:
  Total text regions processed: 12996
  Valid crops created: 12995
  Total augmented crops: 51980
  Images without box files: 0
  Invalid/skipped regions: 1
  Success rate: 100.0%

--- Processing Validation Set ---


Processing img data: 100%|██████████| 50/50 [00:13<00:00,  3.71it/s]


Processing stats for img:
  Total text regions processed: 2746
  Valid crops created: 2746
  Total augmented crops: 2746
  Images without box files: 0
  Invalid/skipped regions: 0
  Success rate: 100.0%

Dataset prepared:
  Training: 12995 samples
  Validation: 2746 samples

--- Checking Data Quality ---
--- Downloading Pretrained Model ---
Extracting to /content/PaddleOCR/pretrain_models/en_ppocr_v3_rec/...


/tmp/ipython-input-1888511781.py:50: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(model_config['extract_dir'])


✅ Successfully downloaded model to /content/PaddleOCR/pretrain_models/en_ppocr_v3_rec/en_PP-OCRv3_rec_train/best_accuracy.pdparams
Pretrained model ready: /content/PaddleOCR/pretrain_models/en_ppocr_v3_rec/en_PP-OCRv3_rec_train/best_accuracy.pdparams
Character dictionary: /content/PaddleOCR/ppocr/utils/en_dict.txt
Configuration created: /content/drive/MyDrive/svtr_training/config.yml
--- Starting Training with Debugging ---
=== QUICK FIX: Replacing MultiLabelEncode with CTCLabelEncode ===
Config fixed! Retrying training...
Starting training...
Command: python3 tools/train.py -c /content/drive/MyDrive/svtr_training/config.yml
[2025/09/07 15:25:16] ppocr INFO: Architecture : 
[2025/09/07 15:25:16] ppocr INFO:     Backbone : 
[2025/09/07 15:25:16] ppocr INFO:         depth : [3, 6, 3]
[2025/09/07 15:25:16] ppocr INFO:         drop_path : 0.02
[2025/09/07 15:25:16] ppocr INFO:         drop_rate : 0.02
[2025/09/07 15:25:16] ppocr INFO:         embed_dim : [64, 128, 256]
[2025/09/07 15:25:16